# GAAE Training — DELCODE DMN-Only (46 Nodes)

This notebook trains the GAAE model on DMN-only correlation matrices (46 nodes)
from `DATA/DELCODE/__v5__/dmn_only_schaefer`.

Key differences from the whole-brain notebook:
- Uses `GraphDMNDatasetInMemoryFiltered` (DMN file suffixes)
- Subject-level 60/20/20 train/val/test split via pre-generated CSVs
- Adjusted hyperparameters (`adjacency_k=8`, `latent_dim=32`)

## Imports

In [ ]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import random
import numpy as np
import torch
from torch_geometric.loader import DataLoader

# Project root
base_dir = Path('/mnt/e/fyassine/ad-early-detection/MODEL')
sys.path.insert(0, str(base_dir))

from model.GAAE.models import GraphAttentionAutoencoderConditioned
from model.GAAE.dataset import GraphDMNDatasetInMemoryFiltered
from model.GAAE.utils import knn_binary_adjacency_matrix_no_diag
from model.GAAE.train import train_model_with_val_notebook_train_loss

## Configuration

In [ ]:
# Weights & Biases (optional — set to None to disable)
WANDB_PROJECT = "ad-early-detection-dmn"

try:
    import wandb
    wandb.login()
except Exception:
    wandb = None
    print("wandb not available — logging disabled")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load hyperparameters
hyperparams_path = base_dir / "configs" / "gaae_delcode_dmn.json"
with open(hyperparams_path, "r") as handle:
    hyperparams = json.load(handle)

seed = hyperparams["seed"]
batch_size = hyperparams["batch_size"]
learning_rate = hyperparams["learning_rate"]
adj_loss_weight = hyperparams["adj_loss_weight"]
n_epochs = hyperparams["epochs"]
early_stopping_patience = hyperparams["early_stopping_patience"]

out_features = hyperparams["latent_dim"]
num_heads = hyperparams["num_heads"]
cond_dim = hyperparams["cond_dim"]
dropout = hyperparams["dropout"]
model_save_path = hyperparams.get("model_save_path")

adjacency_args = {"k": hyperparams["adjacency_k"]}
num_workers = hyperparams["num_workers"]
file_variant = hyperparams.get("file_variant", "z_transformed")

print(f"Hyperparameters: {json.dumps(hyperparams, indent=2)}")

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(seed)

def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(f"Random seed set to: {seed}")

## Dataset

Uses pre-generated `train.csv`, `val.csv`, `test.csv` for subject-level splits.
Each CSV has a `Repseudonym` column listing the subject IDs in that split.

In [ ]:
# DMN data root
dmn_data_root = "/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v5__/dmn_only_schaefer"

# Split CSV paths (generated by generate_dmn_splits.py)
train_csv = os.path.join(dmn_data_root, "train.csv")
val_csv   = os.path.join(dmn_data_root, "val.csv")
test_csv  = os.path.join(dmn_data_root, "test.csv")

# Create datasets for each split
train_dataset = GraphDMNDatasetInMemoryFiltered(
    root=dmn_data_root,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    filter_csv_path=train_csv,
    separator=",",
    file_variant=file_variant,
)

val_dataset = GraphDMNDatasetInMemoryFiltered(
    root=dmn_data_root,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    filter_csv_path=val_csv,
    separator=",",
    file_variant=file_variant,
)

test_dataset = GraphDMNDatasetInMemoryFiltered(
    root=dmn_data_root,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    filter_csv_path=test_csv,
    separator=",",
    file_variant=file_variant,
)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"Feature shape: {train_dataset[0].x.shape}")

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    worker_init_fn=worker_init_fn,
    persistent_workers=True if num_workers > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    worker_init_fn=worker_init_fn,
    persistent_workers=True if num_workers > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

dataset_info = {
    "dataset_name": "DMN-only Schaefer 46-node (60/20/20 subject-level split)",
    "kNN_param": adjacency_args['k'],
    "correlation_type": file_variant,
    "num_features": train_dataset[0].x.size(1),
    "train_dataset_size": len(train_dataset),
    "val_dataset_size": len(val_dataset),
    "test_dataset_size": len(test_dataset),
    "batch_size": batch_size
}
print(f"Dataset info: {json.dumps(dataset_info, indent=2)}")

## Model

In [ ]:
# Derive in_features from the dataset
in_features = train_dataset[0].x.size(1)
hidden_dim = in_features  # same as in_features by default

model = GraphAttentionAutoencoderConditioned(
    in_features=in_features,
    hidden_dim=hidden_dim,
    out_features=out_features,
    cond_dim=cond_dim,
    num_heads=num_heads,
    dropout=dropout
).to(device)

model_config = {
    "model_type": model.__class__.__name__,
    "in_features": in_features,
    "hidden_size": hidden_dim,
    "latent_dim": out_features,
    "attention_heads": num_heads,
    "device": device.type,
    "dropout": dropout
}

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(f"Model: {model.__class__.__name__}")
print(f"  in_features={in_features}, hidden_dim={hidden_dim}, out_features={out_features}")
print(f"  num_heads={num_heads}, dropout={dropout}")

## Training

In [ ]:
best_model, history = train_model_with_val_notebook_train_loss(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    batch_size=batch_size,
    learning_rate=learning_rate,
    model_config=model_config,
    adj_loss_weight=adj_loss_weight,
    epochs=n_epochs,
    early_stopping_patience=early_stopping_patience,
    dataset_info=dataset_info,
    project_name=WANDB_PROJECT
)

# Save model
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
try:
    wandb_run_name = wandb.run.name if wandb and wandb.run and wandb.run.name else f"run_{run_timestamp}"
except:
    wandb_run_name = f"run_{run_timestamp}"

run_artifact_dir = os.path.join("checkpoints", wandb_run_name)
os.makedirs(run_artifact_dir, exist_ok=True)

model_file = os.path.join(run_artifact_dir, f"model_{wandb_run_name}.pth")
torch.save(best_model, model_file)
print(f"Saved best model to {model_file}")

config_to_save = {
    "run_name": wandb_run_name,
    "timestamp": run_timestamp,
    "dataset_info": dataset_info,
    "model_config": model_config,
    "training_config": {
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "adj_loss_weight": adj_loss_weight,
        "epochs": n_epochs,
        "early_stopping_patience": early_stopping_patience
    }
}

def json_serial(obj):
    if isinstance(obj, (datetime, torch.device)):
        return str(obj)
    raise TypeError(f"Type {type(obj)} not serializable")

config_file = os.path.join(run_artifact_dir, "run_config.json")
with open(config_file, "w") as f:
    json.dump(config_to_save, f, indent=4, default=json_serial)
print(f"Saved run configuration to {config_file}")

## Loss Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss (DMN 46-node)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()